In [ ]:
from importlib import import_module
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/extracted').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

evaluation = import_module('src.steps.03_evaluation')
EvaluationItem, Evaluator = evaluation.EvaluationItem, evaluation.Evaluator
SAMPLE_SIZE, SEED, WORKERS = 5, 42, 1

In [ ]:
def load_items(limit=1, seed=42):
    extracted = PROJECT_ROOT / '.data/extracted'
    annotated = PROJECT_ROOT / '.data/annotated'
    texts = {p.relative_to(extracted).with_suffix('').as_posix(): p for p in extracted.rglob('*.txt')}
    labels = {p.relative_to(annotated).with_suffix('').as_posix(): p for p in annotated.rglob('*.json')}
    pairs = sorted(texts.keys() & labels.keys())
    if not pairs:
        raise FileNotFoundError('No matching resume text/JSON pairs found.')
    if limit is not None:
        pairs = random.Random(seed).sample(pairs, min(limit, len(pairs)))

    items = []
    for pair in pairs:
        target = json.loads(labels[pair].read_text(encoding='utf-8'))
        items.append(EvaluationItem(
            resume_id=Path(pair).stem,
            category=Path(pair).parent.as_posix(),
            resume_text=texts[pair].read_text(encoding='utf-8'),
            target_json=target,
        ))
    return items, len(texts.keys() & labels.keys())

In [ ]:
items, available_pairs = load_items(SAMPLE_SIZE, SEED)
print(f'Available paired resumes: {available_pairs:,}')
print(f'Loaded for this test: {len(items)}')

In [ ]:
evaluator = Evaluator(items)

In [ ]:
print(evaluation.render_json_prompt(items[0].resume_text))

In [ ]:
items[0]

In [ ]:
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

ResumeOutput = import_module('src.schema.resume_output').ResumeOutput
ModelCallbackOutput = evaluation.ModelCallbackOutput
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'llama3.2'
OLLAMA_TIMEOUT = 300

def ollama_inference(item):
    prompt = evaluation.render_json_prompt(item.resume_text)
    payload = {
        'model': OLLAMA_MODEL,
        'prompt': prompt,
        'stream': False,
        'format': ResumeOutput.model_json_schema(),
        'options': {'temperature': 0},
        'keep_alive': '5m',
    }
    request = Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urlopen(request, timeout=OLLAMA_TIMEOUT) as response:
            result = json.loads(response.read().decode('utf-8'))
    except HTTPError as error:
        detail = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Ollama returned HTTP {error.code}: {detail}') from error
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            f'Cannot reach Ollama at {OLLAMA_URL}. Start Ollama and confirm {OLLAMA_MODEL} is available.'
        ) from error

    if result.get('error'):
        raise RuntimeError(f'Ollama error: {result["error"]}')
    if 'response' not in result:
        raise RuntimeError('Ollama response is missing its generated text.')
    return ModelCallbackOutput(
        raw_output=result['response'],
        prompt=prompt,
        model_name=OLLAMA_MODEL,
        metadata={
            'total_duration_ns': result.get('total_duration'),
            'prompt_eval_count': result.get('prompt_eval_count'),
            'eval_count': result.get('eval_count'),
        },
    )

In [ ]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'
OPENROUTER_TIMEOUT = 300

def openrouter_inference(model_name):
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        raise RuntimeError('OPENROUTER_API_KEY is missing from the environment or project .env.')

    def callback(item):
        prompt = evaluation.render_json_prompt(item.resume_text)

        payload = {
            'model': model_name,
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0,
            'response_format': {
                'type': 'json_schema',
                'json_schema': {
                    'name': 'resume_output',
                    'strict': True,
                    'schema': evaluation.strict_json_schema(ResumeOutput.model_json_schema()),
                },
            },
        }
        request = Request(
            OPENROUTER_URL,
            data=json.dumps(payload).encode('utf-8'),
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {api_key}',
            },
            method='POST',
        )
        try:
            with urlopen(request, timeout=OPENROUTER_TIMEOUT) as response:
                result = json.loads(response.read().decode('utf-8'))
        except HTTPError as error:
            detail = error.read().decode('utf-8', errors='replace')
            raise RuntimeError(f'OpenRouter returned HTTP {error.code}: {detail}') from error
        except (URLError, TimeoutError) as error:
            raise RuntimeError(f'Cannot reach OpenRouter at {OPENROUTER_URL}: {error}') from error

        if result.get('error'):
            raise RuntimeError(f'OpenRouter error: {result["error"]}')
        choices = result.get('choices') or []
        content = choices[0].get('message', {}).get('content') if choices else None
        if not content:
            raise RuntimeError('OpenRouter response is missing its generated text.')
        return ModelCallbackOutput(
            raw_output=content,
            prompt=prompt,
            model_name=result.get('model', model_name),
            metadata=result.get('usage', {}),
        )

    return callback


In [ ]:

# results = evaluator.evaluate(
#     callback=ollama_inference,
#     workers=WORKERS,
# )

In [ ]:
# for result in results.results:
#     print({
#         'resume_id': result.item.resume_id,
#         'overall': None if result.overall_score is None else round(result.overall_score * 100, 2),
#         'programmatic': None if result.programmatic_score is None else round(result.programmatic_score * 100, 2),
#         'semantic': None if result.semantic_score is None else round(result.semantic_score * 100, 2),
#         'identity': None if result.identity_score is None else round(result.identity_score * 100, 2),
#     })

# evaluation.report_results(results)


In [ ]:
result = openrouter_inference("openai/gpt-5.6-luna")

In [ ]:
result(items[0])

In [ ]:
OPENROUTER_MODELS = [
    'google/gemini-3.1-flash-lite',
    'openai/gpt-5.6-luna',
]

reports = {}
for model_name in OPENROUTER_MODELS:
    reports[f'openrouter:{model_name}'] = evaluator.evaluate(
        callback=openrouter_inference(model_name),
        workers=WORKERS,
    )

evaluation.report_results(reports)
